# Break-point prediction dataset

Combines every `match_snapshots*.csv` under `data/logs/`, filters to **break-point states**
(receiver on game point in regular games; **all tiebreak points** included as mini-break
opportunities, flagged via `is_tiebreak`), labels each with whether the receiver won that
point, and builds server/receiver-oriented features for modeling.

Label: `converted = 1` if the receiver won the point (break / mini-break), `0` if the server saved it.

In [2]:
import glob
import numpy as np
import pandas as pd

ROOT = r"data/logs"
files = glob.glob(ROOT + r"/**/match_snapshots*.csv", recursive=True)
frames = []
for f in files:
    try:
        d = pd.read_csv(f, parse_dates=["timestamp"])
        if "kalshi_p1_ask" in d.columns:      # new-schema files only
            d["source_file"] = f
            frames.append(d)
    except Exception as e:
        print(f"skip {f}: {e}")
snap = (pd.concat(frames, ignore_index=True)
          .dropna(subset=["kalshi_p1_ask", "kalshi_p2_ask"])
          .drop_duplicates(subset=["ticker", "timestamp"]))
snap["mid"] = (snap.kalshi_p1_ask + snap.kalshi_p1_bid) / 2
snap["evk"] = snap.ticker.str.rsplit("-", n=1).str[0]
snap = snap.sort_values(["evk", "timestamp"]).reset_index(drop=True)
print(f"{len(snap)} snapshots, {snap.evk.nunique()} matches, {len(frames)} files")

4199 snapshots, 29 matches, 25 files


C:\Users\agran\AppData\Local\Temp\ipykernel_48764\2568736273.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["source_file"] = f
C:\Users\agran\AppData\Local\Temp\ipykernel_48764\2568736273.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["source_file"] = f
C:\Users\agran\AppData\Local\Temp\ipykernel_48764\2568736273.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.con

In [3]:
# ---- filter to break-point states and label the outcome of that point ----
_NOTE = {"0": 0, "15": 1, "30": 2, "40": 3, "Ad": 4, "AD": 4}

def parse_state(score_str, game_score_str):
    """-> (n_sets, games_p1, games_p2, in_tb, pts_p1, pts_p2) or None."""
    try:
        parts = str(score_str).split()
        ga, gb = map(int, parts[-1].split("-"))
        in_tb = ga == 6 and gb == 6
        x, y = str(game_score_str).split("-")
        if in_tb:
            return len(parts), ga, gb, True, int(x), int(y)
        return len(parts), ga, gb, False, _NOTE[x], _NOTE[y]
    except Exception:
        return None

rows = []
for evk, g in snap.groupby("evk"):
    g = g.reset_index(drop=True)
    for i in range(len(g) - 1):
        r, nxt = g.iloc[i], g.iloc[i + 1]
        st, st2 = parse_state(r.score_str, r.game_score_str), parse_state(nxt.score_str, nxt.game_score_str)
        if st is None or st2 is None or st[0] != st2[0]:
            continue
        if (nxt.timestamp - r.timestamp).total_seconds() > 240:   # feed gap
            continue
        ns, ga, gb, in_tb, pa, pb = st
        srv_p1 = r.server == "p1"
        srv_pts, ret_pts = (pa, pb) if srv_p1 else (pb, pa)
        # break-point state? regular game: receiver on game point. tiebreak: every point counts.
        if not in_tb:
            is_bp = ret_pts >= 3 and ret_pts - srv_pts >= 1
        else:
            is_bp = True
        if not is_bp:
            continue
        # label: did the receiver win the NEXT point?
        converted = None
        if in_tb:
            if st2[3] and (st2[4] + st2[5]) == (pa + pb) + 1:          # next TB point
                p1_won_pt = st2[4] == pa + 1
                converted = int(p1_won_pt != srv_p1)
            elif not st2[3] and (st2[1] + st2[2]) == 13:               # TB just ended (7-6/6-7 -> new set not yet)
                converted = None                                        # ambiguous boundary, drop
        else:
            if not st2[3] and (st2[1] + st2[2]) == (ga + gb) + 1:      # game concluded
                p1_won_game = st2[1] == ga + 1
                converted = int(p1_won_game != srv_p1)                  # receiver won -> break
            elif not st2[3] and (st2[1], st2[2]) == (ga, gb):          # same game continues -> BP saved
                converted = 0
        if converted is None:
            continue
        rows.append({
            "evk": evk, "timestamp": r.timestamp, "is_tiebreak": in_tb,
            "converted": converted,
            "server_is_p1": srv_p1, "sets_played": ns - 1,
            "games_srv": ga if srv_p1 else gb, "games_ret": gb if srv_p1 else ga,
            "pts_srv": srv_pts, "pts_ret": ret_pts,
            "row_idx": i,
        })
bp = pd.DataFrame(rows)
print(f"break-point states: {len(bp)}   (regular {len(bp[~bp.is_tiebreak])}, tiebreak {len(bp[bp.is_tiebreak])})")
print(f"conversion rate: overall {bp.converted.mean()*100:.1f}%  |  regular {bp[~bp.is_tiebreak].converted.mean()*100:.1f}%  |  TB {bp[bp.is_tiebreak].converted.mean()*100:.1f}%")

break-point states: 506   (regular 306, tiebreak 200)
conversion rate: overall 35.6%  |  regular 36.6%  |  TB 34.0%


In [4]:

# ---- ATP rating points as of match day ----
import sqlite3, re, unicodedata
_con = sqlite3.connect(r"atp/data/atp.db")
def _norm(n):
    n = unicodedata.normalize("NFD", str(n))
    n = "".join(c for c in n if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z]", "", n.lower())
_pidx = {_norm((a or "") + (b or "")): p for p, a, b in _con.execute("SELECT player_id, first_name, last_name FROM players")}
_pts_cache = {}
def rating_points(name, matchdate):
    key = (_norm(name), str(matchdate.date()))
    if key in _pts_cache: return _pts_cache[key]
    pid = _pidx.get(_norm(name)); v = None
    if pid is not None:
        r = _con.execute("SELECT roll_points FROM player_rankings WHERE player_id=? AND rank_date<=? ORDER BY rank_date DESC LIMIT 1",
                         (pid, str(matchdate.date()))).fetchone()
        v = r[0] if r else None
    v = max(v, 10) if v else 10          # unranked/missing -> floor of 10 pts
    _pts_cache[key] = v
    return v

# ---- server/receiver-oriented features at the break-point moment ----
STATS = ["first_serve", "first_serve_won", "second_serve_won", "first_return_won", "second_return_won"]

def features(row):
    r = snap[snap.evk == row.evk].iloc[row.row_idx]   # original snapshot
    s, o = ("p1", "p2") if row.server_is_p1 else ("p2", "p1")
    f = {}
    for st in STATS:
        for side, tag in [(s, "srv"), (o, "ret")]:
            num, den = r[f"{side}_{st}_num"], r[f"{side}_{st}_den"]
            f[f"{tag}_{st}"] = num / den if den > 0 else np.nan
            f[f"{tag}_{st}_den"] = den
    f["srv_last10"] = r[f"{s}_last10_pts_won"]
    f["ret_last10"] = r[f"{o}_last10_pts_won"]
    f["srv_price"] = r.mid if row.server_is_p1 else 1 - r.mid
    f["spread"] = r.kalshi_p1_ask - r.kalshi_p1_bid
    f["mc_match_srv"] = r.mc_prob_p1 if row.server_is_p1 else 1 - r.mc_prob_p1
    f["mc_game_srv"] = r.mc_game_prob_p1 if row.server_is_p1 else 1 - r.mc_game_prob_p1
    sp = rating_points(r[f"{s}_name"], r.timestamp)
    rp = rating_points(r[f"{o}_name"], r.timestamp)
    f["log_pts_ratio"] = np.log(sp / rp)
    return pd.Series(f)

X = bp.join(bp.apply(features, axis=1))
X.to_csv("data/breakpoint_dataset.csv", index=False)
print(f"dataset: {X.shape[0]} rows x {X.shape[1]} cols -> data/breakpoint_dataset.csv")
X.head()

dataset: 506 rows x 38 cols -> data/breakpoint_dataset.csv


,evk,timestamp,is_tiebreak,converted,server_is_p1,sets_played,games_srv,games_ret,pts_srv,pts_ret,...,srv_second_return_won_den,ret_second_return_won,ret_second_return_won_den,srv_last10,ret_last10,srv_price,spread,mc_match_srv,mc_game_srv,log_pts_ratio
0,KXATPCHALLENGERMATCH-26JUL03LEGBAR,2026-07-04 00:56:13,False,0,False,0,0,3,1,3,...,4.0,0.714286,7.0,NaN,NaN,0.175,0.01,0.1826,0.2585,-0.805625
1,KXATPCHALLENGERMATCH-26JUL03LEGBAR,2026-07-04 00:56:47,False,0,False,0,0,3,2,3,...,4.0,0.750000,8.0,3.0,7.0,0.185,0.01,0.1723,0.3971,-0.805625
2,KXATPCHALLENGERMATCH-26JUL03LEGBAR,2026-07-04 01:01:41,False,1,True,0,3,1,0,3,...,10.0,0.400000,5.0,3.0,7.0,0.655,0.01,0.6491,0.1893,0.805625
3,KXATPCHALLENGERMATCH-26JUL03LEGBAR,2026-07-04 01:06:28,False,1,False,0,2,3,2,3,...,6.0,0.600000,10.0,7.0,3.0,0.325,0.01,0.3680,0.4303,-0.805625
4,KXATPCHALLENGERMATCH-26JUL03LEGBAR,2026-07-04 01:12:26,False,0,True,0,4,2,3,4,...,10.0,0.555556,9.0,6.0,4.0,0.685,0.01,0.6395,0.4223,0.805625


In [6]:
# ---- baseline models: does anything beat the base rate? ----
# Split BY MATCH (never by row: leakage), but choose test matches greedily so the
# ROW split lands near 80/20 regardless of how many BPs each match produced.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier   # swap for xgboost if installed
from sklearn.metrics import roc_auc_score, log_loss

feat_cols = ["srv_first_serve", "srv_first_serve_won", "srv_second_serve_won",
             "ret_first_return_won", "ret_second_return_won", "srv_last10", 
            #  "srv_price", 
             "log_pts_ratio"]
print(f"features ({len(feat_cols)}):")
print("  " + ", ".join(feat_cols))

d = X.dropna(subset=feat_cols)
rng = np.random.default_rng(7)
order = rng.permutation(d.evk.unique())
sizes = d.evk.value_counts()
test_matches, acc, target = set(), 0, int(0.20 * len(d))
for m in order:
    if acc >= target: break
    test_matches.add(m); acc += sizes[m]
tr, te = d[~d.evk.isin(test_matches)], d[d.evk.isin(test_matches)]
print(f"\ntrain {len(tr)} rows / {d.evk.nunique() - len(test_matches)} matches;  test {len(te)} rows / {len(test_matches)} matches ({len(te)/len(d)*100:.0f}%)")
print(f"base rate: train {tr.converted.mean()*100:.1f}%  test {te.converted.mean()*100:.1f}%\n")

base_ll = log_loss(te.converted, np.full(len(te), tr.converted.mean()))
print(f"{'model':<24} {'test AUC':>8} {'test logloss':>12}   (base-rate logloss {base_ll:.4f})")
ratio = (1 - tr.converted.mean()) / tr.converted.mean()
for name, m in [("logistic", LogisticRegression(max_iter=2000, C=0.5)),
                 ("logistic BALANCED", LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced")),
                 ("xgboost", XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
                                          subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                                          eval_metric="logloss", verbosity=0)),
                 ("xgboost WEIGHTED", XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
                                          subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                                          scale_pos_weight=ratio, eval_metric="logloss", verbosity=0)),
                 ("random forest", RandomForestClassifier(n_estimators=400, max_depth=4,
                                                          min_samples_leaf=15, random_state=0)),
                 ("random forest BALANCED", RandomForestClassifier(n_estimators=400, max_depth=4,
                                                          min_samples_leaf=15, random_state=0, class_weight="balanced"))]:
    m.fit(tr[feat_cols], tr.converted)
    p = m.predict_proba(te[feat_cols])[:, 1]
    print(f"{name:<24} {roc_auc_score(te.converted, p):>8.3f} {log_loss(te.converted, p):>12.4f}   mean predicted p: {p.mean()*100:5.1f}%  (actual {te.converted.mean()*100:.1f}%)")

# stability check: repeat over 10 random match-level 80/20 splits
aucs = []
for seed in range(10):
    rng = np.random.default_rng(seed)
    order = rng.permutation(d.evk.unique())
    tm, acc = set(), 0
    for m2 in order:
        if acc >= target: break
        tm.add(m2); acc += sizes[m2]
    tr2, te2 = d[~d.evk.isin(tm)], d[d.evk.isin(tm)]
    if te2.converted.nunique() < 2: continue
    lm = LogisticRegression(max_iter=2000, C=0.5).fit(tr2[feat_cols], tr2.converted)
    aucs.append(roc_auc_score(te2.converted, lm.predict_proba(te2[feat_cols])[:, 1]))
print(f"\nlogistic AUC over 10 match-level splits: mean {np.mean(aucs):.3f}  min {min(aucs):.3f}  max {max(aucs):.3f}")

# ---- what the logistic actually learned (standardized coefficients) ----
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(d[feat_cols])
lm = LogisticRegression(max_iter=2000, C=0.5).fit(sc.transform(d[feat_cols]), d.converted)
print("\nstandardized coefficients (+ = more likely the break CONVERTS):")
for name, c in sorted(zip(feat_cols, lm.coef_[0]), key=lambda x: -abs(x[1])):
    bar = "#" * int(abs(c) * 40)
    print(f"  {name:<22} {c:+.3f}  {bar}")


features (7):
  srv_first_serve, srv_first_serve_won, srv_second_serve_won, ret_first_return_won, ret_second_return_won, srv_last10, log_pts_ratio

train 392 rows / 21 matches;  test 113 rows / 7 matches (22%)
base rate: train 34.7%  test 38.9%

model                    test AUC test logloss   (base-rate logloss 0.6724)
logistic                    0.486       0.6906   mean predicted p:  35.0%  (actual 38.9%)
logistic BALANCED           0.487       0.7128   mean predicted p:  49.9%  (actual 38.9%)
xgboost                     0.516       0.7093   mean predicted p:  37.7%  (actual 38.9%)
xgboost WEIGHTED            0.513       0.7344   mean predicted p:  50.3%  (actual 38.9%)
random forest               0.475       0.6850   mean predicted p:  36.2%  (actual 38.9%)
random forest BALANCED      0.465       0.7039   mean predicted p:  49.0%  (actual 38.9%)

logistic AUC over 10 match-level splits: mean 0.508  min 0.444  max 0.548

standardized coefficients (+ = more likely the break CONVERTS)

**Reading the result:** the model earns its existence only if test AUC is meaningfully above 0.5
**and** test log-loss beats the base-rate log-loss. With ~20 matches the test set is small - shuffle
`rng` seeds / use grouped CV before believing anything. If it does show signal, the next question is
whether the signal survives what the market already prices (add `srv_price`-only model as comparison).